# 04 · Forecast: what private BEV share is plausible by 2028?

**Business question:** How fast is Sweden shifting to battery-electric cars among private buyers, where is it fastest or slowest and why, and what BEV share is plausible by 2028?

This notebook answers the last part, **what is plausible by 2028**, for the national private-buyer BEV share. It builds on the earlier notebooks:
- **01:** the private share is strongly seasonal (Sep and Dec spikes). 2026 is up 9 pp on 2025 (Jan–Aug), driven by BEV volume.
- **02:** counties move together, and regional splits are too noisy for a 2028 horizon. So this forecast is national.
- **03:** ending the climate bonus in Nov 2022 broke the trend. Growth went from +1 pp/month to flat. Monthly residuals are strongly autocorrelated. 03 recommended fitting on the post-bonus period only and presenting **scenarios rather than one line**.

**Rules from `CLAUDE.md`:**
- Shares stay within 0–1, and every forecast comes with intervals or scenarios.
- Annual shares are ratios of sums, never averages of monthly shares.
- Data runs through **Aug 2026**. The horizon is **Dec 2028** (28 months ahead).

## 1. Setup and data

**Choices made here**
- **Private buyers only**, built from the county fact table exactly as in 01 and 03. County 99 is included, because this is the national total.
- **Training window: Jan 2023 – Aug 2026** (44 months). 03 showed the bonus era (to Aug 2022) and the transition months (Sep–Dec 2022) belong to a different regime. Including them would carry the bonus-era growth rate of +1 pp/month into the forecast.
- **Logit transform**, $z_t = \log\big(s_t / (1 - s_t)\big)$. Every model forecasts $z$, and the forecasts are converted back with $s = 1/(1+e^{-z})$. Every forecast, and every simulated path, therefore stays strictly between 0 and 1. A constant trend in $z$ is also an S-curve in $s$: growth is fastest near 50% and slows towards 0% and 100%, which is how adoption curves usually behave.
- **Monthly data with month-of-year effects**, rather than a 12-month rolling share. The rolling share is smooth but lags by about six months (see 01), and the seasonal spikes are large enough to need modelling.

In [1]:
import sys
import logging
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.structural import UnobservedComponents

warnings.filterwarnings("ignore", message="IProgress not found")    # tqdm notice triggered by importing Prophet
logging.getLogger("prophet.plot").setLevel(logging.CRITICAL)           # "plotly not installed": only static plots are used
from prophet import Prophet
logging.getLogger("prophet").setLevel(logging.CRITICAL)
logging.getLogger("cmdstanpy").disabled = True                        # Stan sampler progress lines

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import viz
from viz import BLUE, ORANGE, AQUA, RED, INK, INK2, MUTED, GRID, AXIS, GREY_DARK, GREY_LIGHT, SURFACE, PCT
viz.apply_style()
PROC, FIGS = ROOT / "data" / "processed", ROOT / "docs" / "figures"
warnings.filterwarnings("ignore", module="statsmodels")   # optimizer chatter; convergence is checked explicitly below

fact = pd.read_csv(PROC / "fact_registrations_county_monthly.csv", parse_dates=["month"], dtype={"county_code": str})
w = fact[fact.owner_type == "private"].pivot_table(index="month", columns="fuel_type", values="registrations",
                                                   aggfunc="sum", fill_value=0)
pv = pd.DataFrame({"bev": w["BEV"], "total": w.sum(axis=1)}).asfreq("MS")
pv["share"] = pv.bev / pv.total
pv["logit"] = np.log(pv.share / (1 - pv.share))

T0, LAST, H_END = pd.Timestamp("2023-01-01"), pv.index.max(), pd.Timestamp("2028-12-01")
HORIZON = pd.date_range(LAST + pd.offsets.MonthBegin(1), H_END, freq="MS")
inv_logit = lambda z: 1 / (1 + np.exp(-z))
assert pv.share.between(0, 1, inclusive="neither").all() and pv.index.is_monotonic_increasing
print(f"Private series {pv.index.min():%b %Y}–{LAST:%b %Y}; training {T0:%b %Y}–{LAST:%b %Y} "
      f"({len(pv.loc[T0:])} months); forecast {HORIZON[0]:%b %Y}–{H_END:%b %Y} ({len(HORIZON)} months)")

Private series Jan 2021–Aug 2026; training Jan 2023–Aug 2026 (44 months); forecast Sep 2026–Dec 2028 (28 months)
